# Experiment 2: Numeric Labels + Mild Sampling

This notebook keeps numeric labels and `max_new_tokens=2`, but uses mild sampling: `temperature=0.1`, `top_p=0.9`.

Models:
1. `google/gemma-4-E4B-it`
2. `Qwen/Qwen3-4B-Instruct-2507`
3. `google/gemma-4-E2B-it`

Tasks:
- AFQMC
- TNEWS
- CMNLI


## 1. Install dependencies

In [ ]:
!pip uninstall -y torchaudio torchvision mlx-vlm || true
!pip install -U transformers datasets accelerate peft trl scikit-learn pandas tqdm sentencepiece


## 2. Configuration

In [1]:
MODELS = {
    "gemma_e4b_it": {
        "model_id": "google/gemma-4-E4B-it",
    },
    "qwen3_4b_instruct_2507": {
        "model_id": "Qwen/Qwen3-4B-Instruct-2507",
    },
    "gemma_e2b_it": {
        "model_id": "google/gemma-4-E2B-it",
    },
}

TASKS = ["afqmc", "tnews", "cmnli"]
SPLIT = "validation"
MAX_SAMPLES = 200
DEBUG_N = 3


## 3. Imports and cleanup

In [2]:
import gc
import os
import re
import time
import warnings

import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")


def cleanup_memory():
    gc.collect()
    try:
        torch.mps.empty_cache()
    except Exception:
        pass
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass


def clear_model(model=None, tokenizer=None):
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    cleanup_memory()


## 4. Task specs

In [3]:
TASK_SPECS = {
    "afqmc": {
        "label_id_to_name": {"0": "不同", "1": "相同"},
        "label_name_to_id": {"不同": "0", "相同": "1"},
        "choice_text": "0=不同\n1=相同",
        "valid_ids": ["0", "1"],
        "label_id_style": "fixed_manual_numeric",
    },
    "cmnli": {
        # Observed HF/CLUE mapping:
        # 0 = neutral, 1 = entailment, 2 = contradiction
        "label_id_to_name": {"0": "中立", "1": "蕴含", "2": "矛盾"},
        "label_name_to_id": {"中立": "0", "蕴含": "1", "矛盾": "2"},
        "choice_text": "0=中立\n1=蕴含\n2=矛盾",
        "valid_ids": ["0", "1", "2"],
        "label_id_style": "fixed_manual_numeric",
    },
    "tnews": {
        "label_id_to_name": {
            "0": "故事", "1": "文化", "2": "娱乐", "3": "体育", "4": "财经",
            "5": "房产", "6": "汽车", "7": "教育", "8": "科技", "9": "国际",
            "10": "旅游", "11": "军事", "12": "股票", "13": "农业", "14": "电竞",
        },
        "label_name_to_id": {
            "故事": "0", "文化": "1", "娱乐": "2", "体育": "3", "财经": "4",
            "房产": "5", "汽车": "6", "教育": "7", "科技": "8", "国际": "9",
            "旅游": "10", "军事": "11", "股票": "12", "农业": "13", "电竞": "14",
        },
        "choice_text": (
            "0=故事\n1=文化\n2=娱乐\n3=体育\n4=财经\n5=房产\n6=汽车\n7=教育\n"
            "8=科技\n9=国际\n10=旅游\n11=军事\n12=股票\n13=农业\n14=电竞"
        ),
        "valid_ids": [str(i) for i in range(15)],
        "label_id_style": "fixed_manual_tnews_numeric",
    },
}


def get_task_spec(task):
    return TASK_SPECS[task]


def label_id_to_name(label_id, spec):
    return spec["label_id_to_name"].get(str(label_id), str(label_id))


def pred_id_to_name(pred_id, spec):
    if pred_id == "__invalid__":
        return "__invalid__"
    return spec["label_id_to_name"].get(str(pred_id), "__invalid__")


## 5. Numeric-label prompts and parser

In [4]:
def build_prompt(task, ex, spec):
    if task == "afqmc":
        return f"""只输出一个数字，不要解释。

{spec["choice_text"]}

句子1：{ex["sentence1"]}
句子2：{ex["sentence2"]}

答案："""

    if task == "cmnli":
        return f"""只输出一个数字，不要解释。

{spec["choice_text"]}

前提：{ex["sentence1"]}
假设：{ex["sentence2"]}

答案："""

    if task == "tnews":
        return f"""只输出一个数字，不要解释。

{spec["choice_text"]}

标题：{ex["sentence"]}

答案："""

    raise ValueError(f"Unsupported task: {task}")


def normalize_output(text):
    text = str(text).strip().lower()

    for token in ["<bos>", "<eos>", "<pad>", "<start_of_turn>", "<end_of_turn>", "model", "assistant", "user"]:
        text = text.replace(token, " ")

    text = text.replace("答案：", " ").replace("答案:", " ")
    text = text.replace("类别：", " ").replace("类别:", " ")
    text = text.replace("标签：", " ").replace("标签:", " ")
    text = text.replace("\n", " ")
    text = re.sub(r"[。，“”，、；;:：\[\]\(\)（）\"'`]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_pred_id(raw_output, spec):
    text_norm = normalize_output(raw_output)

    # Exact output such as "0" or "14"
    if text_norm in spec["valid_ids"]:
        return text_norm

    # First number in output, e.g. "答案是 1"
    m = re.search(r"\b\d+\b", text_norm)
    if m:
        candidate = m.group(0)
        if candidate in spec["valid_ids"]:
            return candidate

    # Fallback: if model outputs label names despite numeric prompt
    for name, label_id in spec["label_name_to_id"].items():
        if name in raw_output:
            return label_id

    return "__invalid__"


## 6. Load and generate

In [5]:
def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

    model.eval()
    return tokenizer, model


@torch.no_grad()
def generate_answer(tokenizer, model, prompt, max_new_tokens=2):
    inputs = tokenizer(prompt, return_tensors="pt")

    try:
        inputs = inputs.to(model.device)
    except Exception:
        pass

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.1,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    return decoded.strip()


## 7. Evaluation loop

In [6]:
def evaluate_loaded_model(model_key, model_id, tokenizer, model):
    model_summaries = []
    model_rows = []

    for task in TASKS:
        dataset = load_dataset("clue", task, split=SPLIT)
        dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
        spec = get_task_spec(task)

        y_true, y_pred = [], []
        start = time.time()

        for idx, ex in enumerate(tqdm(dataset, desc=f"{model_key}/{task}")):
            prompt = build_prompt(task, ex, spec)
            raw = generate_answer(tokenizer, model, prompt, max_new_tokens=2)

            gold_id = str(ex["label"])
            pred_id = extract_pred_id(raw, spec)

            y_true.append(gold_id)
            y_pred.append(pred_id)

            row = {
                "model_key": model_key,
                "model_id": model_id,
                "task": task,
                "split": SPLIT,
                "idx": idx,
                "gold_id": gold_id,
                "gold_name": label_id_to_name(gold_id, spec),
                "raw": repr(raw),
                "raw_normalized": normalize_output(raw),
                "pred_name": pred_id_to_name(pred_id, spec),
                "pred_id": pred_id,
            }
            model_rows.append(row)

            if idx < DEBUG_N:
                print(row)

        elapsed = time.time() - start
        invalid_count = sum(p == "__invalid__" for p in y_pred)
        y_pred_for_score = [p if p != "__invalid__" else "-1" for p in y_pred]

        summary = {
            "model_key": model_key,
            "model_id": model_id,
            "task": task,
            "split": SPLIT,
            "samples": len(y_true),
            "accuracy": accuracy_score(y_true, y_pred_for_score),
            "macro_f1": f1_score(y_true, y_pred_for_score, average="macro", zero_division=0),
            "invalid_rate": invalid_count / len(y_pred),
            "invalid_count": invalid_count,
            "seconds": elapsed,
            "samples_per_second": len(y_true) / elapsed if elapsed > 0 else None,
            "label_id_style": spec["label_id_style"],
        }

        print(summary)
        model_summaries.append(summary)

    return model_summaries, model_rows


def evaluate_one_model(model_key, model_cfg):
    model_id = model_cfg["model_id"]
    cleanup_memory()
    print(f"\n=== Loading {model_key}: {model_id} ===")

    tokenizer = None
    model = None
    try:
        tokenizer, model = load_model(model_id)
        summaries, rows = evaluate_loaded_model(model_key, model_id, tokenizer, model)
    finally:
        print(f"Clearing model from memory: {model_key}")
        clear_model(model, tokenizer)

    return summaries, rows


## 8. Run benchmark

In [7]:
all_summaries = []
all_rows = []

for model_key, model_cfg in MODELS.items():
    summaries, rows = evaluate_one_model(model_key, model_cfg)
    all_summaries.extend(summaries)
    all_rows.extend(rows)

summary_df = pd.DataFrame(all_summaries)
detail_df = pd.DataFrame(all_rows)

summary_df



=== Loading gemma_e4b_it: google/gemma-4-E4B-it ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

gemma_e4b_it/afqmc:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'afqmc', 'split': 'validation', 'idx': 0, 'gold_id': '0', 'gold_name': '不同', 'raw': "'双十一'", 'raw_normalized': '双十一', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'afqmc', 'split': 'validation', 'idx': 1, 'gold_id': '0', 'gold_name': '不同', 'raw': "''", 'raw_normalized': '', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'afqmc', 'split': 'validation', 'idx': 2, 'gold_id': '1', 'gold_name': '相同', 'raw': "''", 'raw_normalized': '', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e4b_it', 'model_id': 'google/gemma-4-E4B-it', 'task': 'afqmc', 'split': 'validation', 'samples': 200, 'accuracy': 0.03, 'macro_f1': 0.048192771084337345, 'invalid_rate': 0.925, 'invalid_count': 185, 'seconds': 73.54113793373108, 'samples_per_second':

...

gemma_e2b_it/tnews:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'tnews', 'split': 'validation', 'idx': 0, 'gold_id': '2', 'gold_name': '娱乐', 'raw': "'江疏'", 'raw_normalized': '江疏', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'tnews', 'split': 'validation', 'idx': 1, 'gold_id': '9', 'gold_name': '国际', 'raw': "'以色列大规模'", 'raw_normalized': '以色列大规模', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'tnews', 'split': 'validation', 'idx': 2, 'gold_id': '4', 'gold_name': '财经', 'raw': "'0='", 'raw_normalized': '0=', 'pred_name': '故事', 'pred_id': '0'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'tnews', 'split': 'validation', 'samples': 200, 'accuracy': 0.005, 'macro_f1': 0.005208333333333333, 'invalid_rate': 0.87, 'invalid_count': 174, 'seconds': 57.332098960876465, 'samples_per_second': 3.4

gemma_e2b_it/cmnli:   0%|          | 0/200 [00:00<?, ?it/s]

{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'cmnli', 'split': 'validation', 'idx': 0, 'gold_id': '0', 'gold_name': '中立', 'raw': "'每个人都很'", 'raw_normalized': '每个人都很', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'cmnli', 'split': 'validation', 'idx': 1, 'gold_id': '1', 'gold_name': '蕴含', 'raw': "'我在很大'", 'raw_normalized': '我在很大', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'cmnli', 'split': 'validation', 'idx': 2, 'gold_id': '2', 'gold_name': '矛盾', 'raw': "'我最'", 'raw_normalized': '我最', 'pred_name': '__invalid__', 'pred_id': '__invalid__'}
{'model_key': 'gemma_e2b_it', 'model_id': 'google/gemma-4-E2B-it', 'task': 'cmnli', 'split': 'validation', 'samples': 200, 'accuracy': 0.005, 'macro_f1': 0.005747126436781609, 'invalid_rate': 0.99, 'invalid_count': 198, 'seconds': 46.574066162109375, 'sam

,model_key,model_id,task,split,samples,accuracy,macro_f1,invalid_rate,invalid_count,seconds,samples_per_second,label_id_style
0,gemma_e4b_it,google/gemma-4-E4B-it,afqmc,validation,200,0.030,0.048193,0.925,185,73.541138,2.719566,fixed_manual_numeric
1,gemma_e4b_it,google/gemma-4-E4B-it,tnews,validation,200,0.045,0.021491,0.715,143,104.057455,1.922015,fixed_manual_tnews_numeric
2,gemma_e4b_it,google/gemma-4-E4B-it,cmnli,validation,200,0.000,0.000000,1.000,200,86.557189,2.310611,fixed_manual_numeric
3,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,afqmc,validation,200,0.340,0.253731,0.000,0,60.398612,3.311334,fixed_manual_numeric
4,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,tnews,validation,200,0.260,0.206515,0.010,2,82.843860,2.414180,fixed_manual_tnews_numeric
5,qwen3_4b_instruct_2507,Qwen/Qwen3-4B-Instruct-2507,cmnli,validation,200,0.650,0.509259,0.000,0,66.315592,3.015882,fixed_manual_numeric
6,gemma_e2b_it,google/gemma-4-E2B-it,afqmc,validation,200,0.120,0.095238,0.815,163,39.815239,5.023202,fixed_manual_numeric
7,gemma_e2b_it,google/gemma-4-E2B-it,tnews,validation,200,0.005,0.005208,0.870,174,57.332099,3.488447,fixed_manual_tnews_numeric
8,gemma_e2b_it,google/gemma-4-E2B-it,cmnli,validation,200,0.005,0.005747,0.990,198,46.574066,4.294235,fixed_manual_numeric


## 9. Inspect outputs

In [ ]:
detail_df.head(20)


In [ ]:
invalid_df = detail_df[detail_df["pred_id"] == "__invalid__"].copy()
print("Invalid count:", len(invalid_df))
invalid_df.head(50)


In [ ]:
# Inspect Gemma E4B-it CMNLI outputs
detail_df[
    (detail_df["model_key"] == "gemma_e4b_it") &
    (detail_df["task"] == "cmnli")
].head(50)


In [ ]:
# Invalid counts by model/task
detail_df.assign(is_invalid=detail_df["pred_id"].eq("__invalid__")).groupby(
    ["model_key", "task"]
)["is_invalid"].agg(["sum", "mean"])


## 10. Save results

In [ ]:
os.makedirs("results", exist_ok=True)

summary_path = "results/chinese_understanding_summary_v14_exp2_numeric_sampling.csv"
detail_path = "results/chinese_understanding_details_v14_exp2_numeric_sampling.csv"

summary_df.to_csv(summary_path, index=False)
detail_df.to_csv(detail_path, index=False)

print("Saved:")
print(summary_path)
print(detail_path)
